In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pygmsh
import gmsh
import trimesh
import netCDF4
import meshio
from scipy.interpolate import RegularGridInterpolator

Import netCDF4-formatted topography file (downloaded from [GEBCO](https://download.gebco.net/))

In [2]:
topofile = 'topo_data/renocarson.nc'
# topofile = 'topo_data/fault155_Ts0Td0Pn0.nc'

In [ ]:
d = netCDF4.Dataset(topofile, 'r')
# print(d.variables)

lat, lon = d.variables['lat'][:], d.variables['lon'][:]
elev = d.variables['elevation'][:]

lon_midpoint = lon[lon.size//2]
lat_midpoint = lat[lat.size//2]
# xx, yy = np.meshgrid(lon,lat)

with open('my_topo.dat','w') as f:
    f.write(f'{lon.size} {lat.size}\n')
    for x in lon:
        # set (0,0) to be center of topo data, then convert to m
        x_meters = (x - lon_midpoint) * 111 * 1000
        f.write(f'{round(x_meters,6)}\n') 
    for y in lat:
        y_meters = (y - lat_midpoint) * 111 * 1000
        f.write(f'{round(y_meters,6)}\n')
    for z in elev.flatten():
        f.write(f'{float(z)}\n')

# elevation = np.flipud(elevation)
# plt.matshow(elevation)
elmin = np.min(elev)
print(f'minimum elevation: {elmin} meters')

minimum elevation: 1189 meters


np.int16(3208)

In [6]:
x_meters = (lon - lon_midpoint) * 111 * 1000
x_meters = np.round(x_meters,5)

y_meters = (lat - lat_midpoint) * 111 * 1000
y_meters = np.round(y_meters,5)

delta_x = x_meters[1] - x_meters[0]
delta_y = y_meters[1] - y_meters[0]

region_x = x_meters[-1] - x_meters[0] - delta_x
region_y = y_meters[-1] - y_meters[0] - delta_y

print(region_x,region_y)
print(x_meters[0],x_meters[-1],y_meters[0],y_meters[-1])
print(delta_x,delta_y)

110075.0 143375.0
-55500.0 55037.5 -72150.0 71687.5
462.5 462.5


In [5]:
gmsh.initialize()
gmsh.model.add("rectangular_box")

# -----------------------------
# Parameters
# -----------------------------
cl = 2000.0

level = 0.0
depth = 30000

fault_cl = 500.0
fault_length = 30e3
fault_width = 15e3
dip = np.radians(60)
z_tor = 1300 # m above sea level

nuc_x, nuc_y = 12e3,0 #coords of nuc patch center
nuc_length = 3e3
nuc_width = 3e3
nuc_cl = 250.0

geo = gmsh.model.geo

# -----------------------------
# Top surface points
# -----------------------------
p1 = geo.addPoint( 0.5*region_x,  0.5*region_y, level,  cl)
p2 = geo.addPoint(-0.5*region_x,  0.5*region_y, level,  cl)
p3 = geo.addPoint(-0.5*region_x, -0.5*region_y, level,  cl)
p4 = geo.addPoint( 0.5*region_x, -0.5*region_y, level,  cl)

# Top edges
l1 = geo.addLine(p1, p2)
l2 = geo.addLine(p2, p3)
l3 = geo.addLine(p3, p4)
l4 = geo.addLine(p4, p1)

# -----------------------------
# Bottom surface points
# -----------------------------
p5 = geo.addPoint( 0.5*region_x,  0.5*region_y, -depth, cl)
p6 = geo.addPoint(-0.5*region_x,  0.5*region_y, -depth, cl)
p7 = geo.addPoint(-0.5*region_x, -0.5*region_y, -depth, cl)
p8 = geo.addPoint( 0.5*region_x, -0.5*region_y, -depth, cl)

# Bottom edges
l5 = geo.addLine(p5, p6)
l6 = geo.addLine(p6, p7)
l7 = geo.addLine(p7, p8)
l8 = geo.addLine(p8, p5)

# Vertical edges
l9  = geo.addLine(p1, p5)
l10 = geo.addLine(p2, p6)
l11 = geo.addLine(p3, p7)
l12 = geo.addLine(p4, p8)

# -----------------------------
# Surfaces
# -----------------------------

# Top
loop1 = geo.addCurveLoop([l1, l2, l3, l4])
s1 = geo.addPlaneSurface([loop1])

# Bottom
loop2 = geo.addCurveLoop([ l5,  l6,  l7,  l8])
s2 = geo.addPlaneSurface([loop2])

# Side 1
loop3 = geo.addCurveLoop([-l4, l12, l8, -l9])
s3 = geo.addPlaneSurface([loop3])

# Side 2
loop4 = geo.addCurveLoop([l9, l5, -l10, -l1])
s4 = geo.addPlaneSurface([loop4])

# Side 3
loop5 = geo.addCurveLoop([l10, l6, -l11, -l2])
s5 = geo.addPlaneSurface([loop5])

# Side 4
loop6 = geo.addCurveLoop([l11, l7, -l12, -l3])
s6 = geo.addPlaneSurface([loop6])


#-----------------------
# Fault points and plane
#-----------------------
p9 = geo.addPoint(0,-fault_length/2,z_tor,fault_cl)
p10 = geo.addPoint(fault_width,-fault_length/2,z_tor,fault_cl)
p11 = geo.addPoint(fault_width,fault_length/2,z_tor,fault_cl)
p12 = geo.addPoint(0,fault_length/2,z_tor,fault_cl)

l13 = geo.addLine(p9,p10)
l14 = geo.addLine(p10,p11)
l15 = geo.addLine(p11,p12)
l16 = geo.addLine(p12,p9)

fault_loop = geo.addCurveLoop([l13,l14,l15,l16])
s7 = geo.addPlaneSurface([fault_loop]) 
# fault label is (2,7) for the 7th 2D object added
geo.rotate([(2,7)],0,0,z_tor,0,1,0,dip)

#--------------------------------------------------
# Nucleation patch (should be coplanar with fault)
#--------------------------------------------------

p13 = geo.addPoint(nuc_x - nuc_width/2, nuc_y - nuc_length/2, z_tor,nuc_cl)
p14 = geo.addPoint(nuc_x + nuc_width/2, nuc_y - nuc_length/2, z_tor,nuc_cl)
p15 = geo.addPoint(nuc_x + nuc_width/2, nuc_y + nuc_length/2, z_tor,nuc_cl)
p16 = geo.addPoint(nuc_x - nuc_width/2, nuc_y + nuc_length/2, z_tor,nuc_cl)

l17 = geo.addLine(p13,p14)
l18 = geo.addLine(p14,p15)
l19 = geo.addLine(p15,p16)
l20 = geo.addLine(p16,p13)

nuc_loop = geo.addCurveLoop([l17,l18,l19,l20])
s8 = geo.addPlaneSurface([nuc_loop]) 
# nuc patch label is (2,8)
geo.rotate([(2,8)],0,0,z_tor,0,1,0,dip)

# -----------------------------
# Physical groups
# -----------------------------
gmsh.model.addPhysicalGroup(2, [s1], 1)
gmsh.model.setPhysicalName(2, 101, "1")

gmsh.model.addPhysicalGroup(2,[s7,s8],3)
gmsh.model.setPhysicalName(2,103,"3")

gmsh.model.addPhysicalGroup(2, [s2, s3, s4, s5, s6],5)
gmsh.model.setPhysicalName(2, 105, "3")

# -----------------------------
# Synchronize + mesh
# -----------------------------
geo.synchronize()

gmsh.write("bbox.geo_unrolled")
print("geometry saved")

gmsh.option.setNumber("Mesh.MshFileVersion", 1.0)

gmsh.model.mesh.generate(3)

gmsh.write("bbox.msh")

gmsh.finalize()

Info    : Writing 'bbox.geo_unrolled'...
geometry saved
Info    : Done writing 'bbox.geo_unrolled'
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Line)
Info    : [ 30%] Meshing curve 5 (Line)
Info    : [ 30%] Meshing curve 6 (Line)
Info    : [ 40%] Meshing curve 7 (Line)
Info    : [ 40%] Meshing curve 8 (Line)
Info    : [ 50%] Meshing curve 9 (Line)
Info    : [ 50%] Meshing curve 10 (Line)
Info    : [ 60%] Meshing curve 11 (Line)
Info    : [ 60%] Meshing curve 12 (Line)
Info    : [ 70%] Meshing curve 13 (Line)
Info    : [ 70%] Meshing curve 14 (Line)
Info    : [ 80%] Meshing curve 15 (Line)
Info    : [ 80%] Meshing curve 16 (Line)
Info    : [ 90%] Meshing curve 17 (Line)
Info    : [ 90%] Meshing curve 18 (Line)
Info    : [100%] Meshing curve 19 (Line)
Info    : [100%] Meshing curve 20 (Line)
Info    : Done meshing 1D (Wall 0.00202371s, CPU 0.001735s)
Info  

Info    : [ 60%] Meshing surface 5 (Plane, Frontal-Delaunay)
Info    : [ 70%] Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : [ 80%] Meshing surface 7 (Plane, Frontal-Delaunay)
Info    : [ 90%] Meshing surface 8 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.224179s, CPU 0.22212s)
Info    : Meshing 3D...
Info    : Done meshing 3D (Wall 1.87499e-05s, CPU 6e-06s)
Info    : 16307 nodes 33194 elements
Info    : Writing 'bbox.msh'...
Info    : Done writing 'bbox.msh'


In [6]:
!./gmsh_plane2topo interpol_topo.in

  -55500.000000000000       -55037.500000000000       -54575.000000000000       -54112.500000000000       -53650.000000000000       -53187.500000000000       -52725.000000000000       -52262.500000000000       -51800.000000000000       -51337.500000000000       -50875.000000000000       -50412.500000000000       -49950.000000000000       -49487.500000000000       -49025.000000000000       -48562.500000000000       -48100.000000000000       -47637.500000000000       -47175.000000000000       -46712.500000000000       -46250.000000000000       -45787.500000000000       -45325.000000000000       -44862.500000000000       -44400.000000000000       -43937.500000000000       -43475.000000000000       -43012.500000000000       -42550.000000000000       -42087.500000000000       -41625.000000000000       -41162.500000000000       -40700.000000000000       -40237.500000000000       -39775.000000000000       -39312.500000000000       -38850.000000000000       -38387.500000000000       -37925.000

In [7]:
gmsh.initialize()

gmsh.merge("bbox_topo.msh")

surf_loop = gmsh.model.geo.add_surface_loop([1,2,3,4,5,6,7])

gmsh.model.geo.synchronize()

volume = gmsh.model.geo.add_volume([surf_loop])

geo.synchronize()
# gmsh.model.addPhysicalGroup(2,[7],tag=103)
#domain
gmsh.model.addPhysicalGroup(3, [volume], tag=1)

gmsh.model.geo.synchronize()

# Enable interpolation from boundaries
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 1)
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)

gmsh.model.mesh.generate(3) # generate 3d mesh

gmsh.model.mesh.optimize("Netgen")

gmsh.write("test_final_basin.msh")

gmsh.finalize()

print("Wrote final optimized mesh")

Info    : Reading 'bbox_topo.msh'...
Info    : 16307 nodes
Info    : 32378 elements
Info    : Done reading 'bbox_topo.msh'
Info    : Meshing 1D...
Info    : Done meshing 1D (Wall 3.74997e-06s, CPU 4e-06s)
Info    : Meshing 2D...
Info    : Done meshing 2D (Wall 1.87922e-05s, CPU 1.6e-05s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 16109 nodes...
Info    : Done tetrahedrizing 16117 nodes (Wall 0.0979271s, CPU 0.096459s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.275261s, CPU 0.259605s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 18.5878 (nodes removed 0 0)
Info    : It. 500 - 500 nodes created - worst tet radius 4.20846 (nodes removed 0 0)
Info    : It. 1000 - 1000 nodes created - worst tet radius 3.3979 (nodes removed 0 0)
Info    : It. 1500 - 1500 node

In [ ]:
#unused 
# # ------------------------------
# # Example basin (to be refined)
# # ------------------------------
# # create triangular surface
# geo = gmsh.model.geo
# p17 = geo.addPoint(-8000,21e3,1000,cl)
# p18 = geo.addPoint(15000,21e3,1000,cl)
# p19 = geo.addPoint(7000,21e3,-12000,cl)

# l21 = geo.addLine(p17,p18)
# l22 = geo.addLine(p18,p19)
# l23 = geo.addLine(p19,p17)

# triangle_cl = geo.addCurveLoop([l21,l22,l23])
# tri_surf = geo.addPlaneSurface([triangle_cl])

# extruded_triangle = geo.extrude([(2,tri_surf)], 0, 20000, 0)

# print(extruded_triangle)
# #basin
# gmsh.model.addPhysicalGroup(3,[2],7)
# gmsh.model.setPhysicalName(3,107,'7')

In [38]:
mesh = meshio.read("step1_new.msh")

# Node coordinates: shape (N, 3)
points = mesh.points.copy()

# -----------------------------------------------------------------------------
# Example topography grid
# -----------------------------------------------------------------------------

lon_midpoint = lon[lon.size//2]
lat_midpoint = lat[lat.size//2]

#latlon centered at (0,0) mesh origin
lon_0 = (lon - lon_midpoint) * 111 * 1000 
lat_0 = (lat - lat_midpoint) * 111 * 1000 

X, Y = np.meshgrid(lon_0,lat_0, indexing='ij')

print(X.shape,Y.shape)

# Transpose topo matrix to match X, Y, and convert to km
Z = elev.T

print(Z.shape)

# Bilinear interpolator
interp = RegularGridInterpolator(
    (lon_0, lat_0),
    Z,
    bounds_error=False,
    fill_value=0.0,
)

# -----------------------------------------------------------------------------
# Find nodes on the top surface
# -----------------------------------------------------------------------------

# Example:
# top surface initially at z = 0
tol = 1e-7

top_nodes = np.where(np.abs(points[:, 2]) < tol)[0]

print(f"Moving {len(top_nodes)} top nodes")

# -----------------------------------------------------------------------------
# Interpolate topography onto mesh nodes
# -----------------------------------------------------------------------------

xy = points[top_nodes, :2]

z_new = interp(xy)

# Move nodes vertically
points[top_nodes, 2] = z_new

# -----------------------------------------------------------------------------
# Create updated mesh
# -----------------------------------------------------------------------------

new_mesh = meshio.Mesh(
    points=points,
    cells=mesh.cells,
    point_data=mesh.point_data,
    cell_data=mesh.cell_data,
    field_data=mesh.field_data,
)


meshio.write(
    "box_topography.msh",
    new_mesh,
    file_format="gmsh22",
)

print("Wrote modified mesh")


(240, 312) (240, 312)
(240, 312)
Moving 86459 top nodes
Wrote modified mesh


In [14]:
m = meshio.read("box_topography.msh")

# Extract triangular faces (trimesh only supports surface meshes)
# Note: .msh files often contain tetrahedrons which trimesh cannot directly 'show'
cells = m.get_cells_type("triangle")

# Create the trimesh object
mesh = trimesh.Trimesh(vertices=m.points, faces=cells)

# Display the mesh in an OpenGL window
# mesh.show()